# EPCLM-A2-State-001 — Persistent State Interaction

Tesla T4、Internet、已有 GITHUB_TOKEN Secret；Save & Run All。
自动执行 16 项正确性检查、四项历史数据 Hash 核验，然后依次从头训练 B1/B2/B3（每组 1M targets / 1000 updates）。
B1 保留原有四头 Token Attention；B2 单头 State Read；B3 同时启用。无 G1 选择门槛，无重复 FFN 实验。
执行 5400 秒、发布 600 秒、安全余量 1200 秒，总计不超过两小时。超时标记 INCOMPLETE。
独立 Test 不访问。单 Seed 不构成科学确认，也不证明超过 128 Token 的长期记忆。
按本次规范明确要求，将最终模型权重与报告发布至独立结果分支；不发布优化器恢复状态。
历史 A2 结果只引用归档记录；原始初始权重没有归档，不宣称严格历史初始化复现。


In [ ]:
from pathlib import Path
import os, signal, subprocess, sys, time

BRANCH = 'codex/epclm-a2-state-001'
REPOSITORY = 'https://github.com/ArcheLabs/mini-cells.git'
WORKSPACE = Path('/kaggle/working')
REPO = WORKSPACE / 'minicells-a2-state-001'
OUTPUT = WORKSPACE / 'epclm-a2-state-001-bootstrap'
HARD_LIMIT_SECONDS = 7200
deadline = time.monotonic() + HARD_LIMIT_SECONDS

def run_bounded(command, cwd=WORKSPACE, timeout_seconds=None):
    remaining = deadline - time.monotonic()
    if timeout_seconds is not None:
        remaining = min(remaining, timeout_seconds)
    if remaining <= 1:
        raise TimeoutError('The two-hour notebook budget is exhausted.')
    process = subprocess.Popen(command, cwd=cwd, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, start_new_session=(os.name == 'posix'))
    try:
        output, _ = process.communicate(timeout=remaining)
    except subprocess.TimeoutExpired:
        if os.name == 'posix': os.killpg(process.pid, signal.SIGTERM)
        else: process.terminate()
        try: output, _ = process.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            if os.name == 'posix': os.killpg(process.pid, signal.SIGKILL)
            else: process.kill()
            output, _ = process.communicate()
        print(output or '')
        raise TimeoutError('Notebook time limit reached; partial reports and final models, if any, remains under ' + str(OUTPUT))
    print(output or '')
    if process.returncode != 0:
        raise RuntimeError('Command failed (' + str(process.returncode) + '). Output is under ' + str(OUTPUT))

if REPO.exists():
    if not (REPO / '.git').exists():
        import shutil
        shutil.rmtree(REPO)
if REPO.exists():
    origin = subprocess.run(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], capture_output=True, text=True)
    if origin.returncode != 0 or 'ArcheLabs/mini-cells' not in origin.stdout:
        raise RuntimeError(f'Refusing to use an unrelated directory: {REPO}')
    run_bounded(['git', '-C', str(REPO), 'fetch', '--depth=1', 'origin', BRANCH])
    run_bounded(['git', '-C', str(REPO), 'switch', '-C', BRANCH, 'FETCH_HEAD'])
    run_bounded(['git', '-C', str(REPO), 'clean', '-fd'])
else:
    run_bounded(['git', 'clone', '--depth=1', '--single-branch', '--branch', BRANCH,
        REPOSITORY, str(REPO)])

SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
OUTPUT = WORKSPACE / ('epclm-a2-state-001-' + SOURCE_COMMIT[:12])

import importlib.util
missing = [name for name in ('datasets','tokenizers','huggingface_hub','pytest') if importlib.util.find_spec(name) is None]
if missing:
    run_bounded([sys.executable, '-m', 'pip', 'install', '--quiet'] + missing, timeout_seconds=600)
execution_seconds = min(5400, deadline - time.monotonic() - 1860)
if execution_seconds < 60:
    raise TimeoutError('Insufficient two-hour budget after setup; training was not started.')
run_bounded([sys.executable, str(REPO / 'research/projects/cellular-language-models/experiments/epclm-a2-state-001/train.py'),
    '--output', str(OUTPUT),
    '--execution-seconds', str(execution_seconds)], cwd=REPO, timeout_seconds=6000)
